## **Olist Data Cleaning and Preparation using Python**

### Objective:
To clean, validate, and prepare the Olist e-commerce datasets for efficient SQL-based analysis and visualization.

In [2]:
# Import Libraries

import pandas as pd
import numpy as np

In [3]:
# Import Dataset

customers=pd.read_csv('olist_customers_dataset.csv')
geolocation=pd.read_csv('olist_geolocation_dataset.csv')
items=pd.read_csv('olist_order_items_dataset.csv')
payments=pd.read_csv('olist_order_payments_dataset.csv')
reviews=pd.read_csv('olist_order_reviews_dataset.csv')
orders=pd.read_csv('olist_orders_dataset.csv')
products=pd.read_csv('olist_products_dataset.csv')
sellers=pd.read_csv('olist_sellers_dataset.csv')
cat_name=pd.read_csv('product_category_name_translation.csv')

### Data Cleaning & Preparation Steps

1. **Audit dataset structure**  
   Examine the number of rows, columns, missing values, and duplicate records to understand the quality and structure of the data.

2. **Validate missing values**  
   Identify and investigate missing values to determine whether they represent valid missing information or require correction.

3. **Validate duplicate records**  
   Check for duplicate rows and duplicate values in important identifier columns before deciding whether any records should be removed.

4. **Remove exact duplicate records**  
   Remove completely identical records where they represent redundant data.

5. **Create derived lookup data**  
   Transform detailed data into a summarized lookup structure where appropriate to make subsequent analysis and joins more efficient.

6. **Validate relationships between datasets**  
   Check key relationships between related datasets to ensure that records can be correctly connected during analysis.

7. **Convert date columns to datetime format**  
   Convert date and timestamp fields into appropriate datetime types for accurate date-based analysis.

8. **Preserve meaningful missing values**  
   Retain missing values when they represent legitimate absence of information rather than artificially filling or removing valid records.

9. **Export prepared datasets**  
   Save the validated and prepared datasets as CSV files for subsequent SQL analysis.

In [4]:
# Extract Datasets shape, missing values and Duplicate rows

dataset={
    'Customers':customers,
    'Geolocation':geolocation,
    'Items':items,
    'Payments':payments,
    'Reviews':reviews,
    'Orders':orders,
    'Products':products,
    'Sellers':sellers,
    'Name_translation':cat_name
}

overview=[]

for name,df in dataset.items():
    overview.append({
        'Dataset':name,
        'Rows':df.shape[0],
        'Columns':df.shape[1],
        'Missing values':df.isna().sum().sum(),
        'Duplicate rows':df.duplicated().sum(),
    })

overview_df=pd.DataFrame(overview)

overview_df
        

,Dataset,Rows,Columns,Missing values,Duplicate rows
0,Customers,99441,5,0,0
1,Geolocation,1000163,5,0,261831
2,Items,112650,7,0,0
3,Payments,103886,5,0,0
4,Reviews,99224,7,145903,0
5,Orders,99441,8,4908,0
6,Products,32951,9,2448,0
7,Sellers,3095,4,0,0
8,Name_translation,71,2,0,0


In [5]:
# Checking data types of columns in all datasets

for name,df in dataset.items():
    print('-' * 60)
    print(name.upper())
    print('-' * 60)
    print(df.dtypes)
   

------------------------------------------------------------
CUSTOMERS
------------------------------------------------------------
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
------------------------------------------------------------
GEOLOCATION
------------------------------------------------------------
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object
------------------------------------------------------------
ITEMS
------------------------------------------------------------
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value         

In [6]:
# Check for missing values in each column of all datasets

for name,df in dataset.items():
    missing=df.isna().sum()
    missing=missing[missing>0]

    print(name.upper())

    if len(missing)==0:
        print('No missing values')
    else:
        print(missing)

    print('-' * 60)
        
    

CUSTOMERS
No missing values
------------------------------------------------------------
GEOLOCATION
No missing values
------------------------------------------------------------
ITEMS
No missing values
------------------------------------------------------------
PAYMENTS
No missing values
------------------------------------------------------------
REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64
------------------------------------------------------------
ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
------------------------------------------------------------
PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
------------

### Geolocation Dataset
        261,831 duplicate rows7

In [7]:
# Check how the duplicated rows are

geolocation[geolocation.duplicated()].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP


In [8]:
# Check for the count of unique values in each column

geolocation.nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [9]:
# Dropping duplicate rows  

geolocation_clean=geolocation.drop_duplicates()

In [10]:
geolocation_clean.shape

(738332, 5)

In [11]:
# Grouping city and state by zip code

geolocation_clean.groupby('geolocation_zip_code_prefix')[
    ['geolocation_city', 'geolocation_state']
].nunique().count()

geolocation_city     19015
geolocation_state    19015
dtype: int64

In [12]:
# Taking mean of all latitude and longitude in each zip code to make one row for each zip code

geo_lookup = geolocation_clean.groupby(
    "geolocation_zip_code_prefix",
    as_index=False
).agg({
    "geolocation_city": "first",
    "geolocation_state": "first",
    "geolocation_lat": "mean",
    "geolocation_lng": "mean"
})

In [13]:
geo_lookup.shape

(19015, 5)

In [14]:
# Load the changes to a new CSV file

geo_lookup.to_csv("geo_lookup.csv", index=False)

### Geolocation Dataset — Final Insight

- The dataset initially contained **1,000,163 rows** with **261,831 exact duplicate records**.
- Duplicate records were removed, resulting in **738,332 unique rows**.
- The dataset contained **19,015 unique ZIP-code prefixes**, which became the basis for creating a ZIP-level lookup.
- Each ZIP-code prefix was validated to have a consistent **city and state mapping**.
- Multiple geographic coordinates existed for many ZIP codes, so the **mean latitude and longitude** were calculated to provide one representative location per ZIP code.
- A new **`geo_lookup`** table was created containing **19,015 rows and 5 columns**: ZIP code prefix, city, state, latitude, and longitude.
- The resulting lookup was exported as **`geo_lookup.csv`** for use in the SQL analysis.

**Final Outcome:**  
The detailed geolocation data was transformed into a compact ZIP-level lookup table that can be efficiently joined with customer and seller data during SQL analysis.

###  Reviews Dataset
        total missing cells       145903
        review_comment_title      87656
        review_comment_message    58247

In [15]:
reviews.shape

(99224, 7)

In [16]:
# Since review_comment_title and review_comment_message has the missing values, checking how many rows 
# has none missing or one missing or both missing

reviews[['review_comment_title','review_comment_message']].isna().sum(axis=1).value_counts()

2    56518
1    32867
0     9839
Name: count, dtype: int64

In [17]:
# Check for no of entries in each of these columns

reviews[['review_comment_title','review_comment_message']].notna().sum()

review_comment_title      11568
review_comment_message    40977
dtype: int64

In [18]:
# Check for number of duplicate review_id rows

reviews['review_id'].duplicated(keep=False).sum()

1603

In [19]:
# Check for number of duplicate order_id rows

reviews['order_id'].duplicated(keep=False).sum()

1098

In [20]:
# Check if duplicate review_id has the same or different order_id

reviews[reviews['review_id'].duplicated(keep=False)].sort_values('review_id')

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [21]:
# Check how duplicated id's are associalted with how many order id's 

reviews[reviews['review_id'].duplicated(keep=False)].groupby('review_id')['order_id'].nunique().value_counts()

order_id
2    764
3     25
Name: count, dtype: int64

In [22]:
# Change datatype of date columns to datetime

review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

reviews[review_date_columns] = reviews[review_date_columns].apply(pd.to_datetime)

### Reviews Dataset — Final Insight

- The dataset contains **99,224 review records** across **7 columns**.
- A total of **145,903 missing cells** were identified, limited to `review_comment_title` and `review_comment_message`.
- **56,518 reviews** have neither a review title nor a review message, **32,867 reviews** have one of the two fields missing, and **9,839 reviews** have both fields available.
- Duplicate `review_id` and `order_id` values were investigated rather than automatically removed. Repeated `review_id` values were associated with different orders, while **no completely duplicate rows** were found.
- The review date columns were converted to the appropriate **datetime** format for accurate time-based analysis.

**Why were the missing values not removed?**

The missing comment fields do not make the review record invalid. A customer can provide a **review score without writing a title or message**. Therefore, removing these rows would unnecessarily discard valid review information such as the review score, order relationship, and review dates.

The missing values were therefore **preserved as NULL/NaN**, allowing them to be handled appropriately during SQL analysis.

**Final Outcome:**  
All **99,224 review records** were retained, while missing comment fields were preserved and date columns were standardized for further analysis.

### Orders Dataset
         total missing cells              4908
         order_approved_at                 160
         order_delivered_carrier_date     1783
        order_delivered_customer_date     2965
              2

In [23]:
# Check for the missing cell columns

orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [24]:
# Split the missing cells across order_status

orders.groupby('order_status')[[
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [25]:
# Filter rows where order_status is deliverd and order_approved_at is null

orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_approved_at'].isna())
][[
    'order_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


In [26]:
# # Filter rows where order_status is deliverd and order_delivered_customer_date is null

orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isna())
][[
    'order_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [27]:
# Change datatype of date columns to datetime

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

### Orders Dataset — Final Insight

- The dataset contains **99,441 order records** across **8 columns**.
- A total of **4,908 missing cells** were identified in the order approval and delivery timestamp columns.
- Missing values were analyzed against `order_status` to understand whether they represented expected or potentially incomplete information.
- Orders in statuses such as **canceled, processing, invoiced, shipped, and unavailable** can legitimately have missing delivery timestamps because they may not have progressed through the complete delivery process.
- Among **delivered orders**, **14 orders** were missing `order_approved_at` and **8 orders** were missing `order_delivered_customer_date`. These records were investigated and found to contain other valid order and delivery information.
- Date and timestamp columns were converted to the appropriate **datetime** format for accurate time-based analysis.

**Why were the missing rows not removed?**

The missing values occur at the **individual field level**, while the corresponding order records remain valid. Removing the entire rows would discard useful information such as the order ID, customer relationship, order status, purchase date, delivery information, and estimated delivery date.

For example, a delivered order with a missing approval timestamp is still a valid delivered order; only the approval timestamp is unavailable. Similarly, an order marked as shipped may not yet have a customer delivery date.

Therefore, the missing timestamps were **preserved as NaN/NULL** rather than deleting valid order records or estimating dates that were not present in the source data.

**Final Outcome:**  
All **99,441 order records** were retained, missing timestamps were preserved, and date columns were standardized for further SQL analysis.

### Products dataset
        total missing cells           2448
        product_category_name          610
        product_name_lenght            610
        product_description_lenght     610
        product_photos_qty             610
        product_weight_g                 2
        product_length_cm                2
        product_height_cm                2
        product_width_cm                 

In [28]:
# Check missing cells distribution among the four columns

products[
    ['product_category_name',
     'product_name_lenght',
     'product_description_lenght',
     'product_photos_qty']
].isna().sum(axis=1).value_counts()

0    32341
4      610
Name: count, dtype: int64

In [29]:
# Check missing cells distribution among the four columns

products[
    ['product_weight_g',
     'product_length_cm',
     'product_height_cm',
     'product_width_cm']
].isna().sum(axis=1).value_counts()

0    32949
4        2
Name: count, dtype: int64

In [30]:
# Check how many product_id in products datset are in items dataset

products['product_id'].isin(items['product_id']).value_counts()

product_id
True    32951
Name: count, dtype: int64

In [31]:
# Check for duplicated product_id

products['product_id'].duplicated().sum()

0

### Products Dataset — Final Insight

- The dataset contains **32,951 product records**.
- A total of **2,448 missing cells** were identified.
- **610 products** have all four descriptive attributes missing: `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`.
- **2 products** have all four physical attributes missing: `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`.
- All **32,951 products** are represented in the `order_items` dataset, confirming that every product has been associated with at least one order.
- `product_id` was checked for duplicates, and **no duplicate product IDs** were found.

**Why were the missing rows not removed?**

The missing values affect only specific product attributes; the corresponding product records are still valid and are used in actual orders. Removing these products would cause product information to be lost when joining `products` with `order_items`.

Since there is no reliable basis for estimating the missing category, descriptive, or physical attributes, the missing values were **preserved as NaN/NULL** rather than artificially filling or deleting valid product records.

**Final Outcome:**  
All **32,951 product records** were retained, missing attributes were preserved, and `product_id` was validated as unique for further SQL analysis.

### Exporting the cleaned dataset for further analysis in SQL.

In [32]:
# Upload all changes to the actual dataset

customers.to_csv("customers.csv", index=False)
orders.to_csv("orders.csv", index=False)
items.to_csv("order_items.csv", index=False)
payments.to_csv("order_payments.csv", index=False)
reviews.to_csv("order_reviews.csv", index=False)
products.to_csv("products.csv", index=False)
sellers.to_csv("sellers.csv", index=False)
cat_name.to_csv("category_translation.csv", index=False)

### Conclusion:
The datasets were successfully validated, cleaned, and standardized while preserving meaningful missing values, and are now ready for SQL analysis.